In [ ]:
%load_ext cudf.pandas

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%%RecordEvent

# %load_ext cudf.pandas
import sys, os
tpch_parent = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, tpch_parent)
# from tpch import load_lineitem, load_orders, load_part, load_nation, load_partsupp, load_supplier, q09
import pandas as pd
DATA_ROOT = "/home/dias-benchmarks/tpch/data"
STORAGE_OPTS = {}

In [ ]:
%Checkpoint /home/dias-benchmarks/tpch/notebooks/q09/q09_rewrite/checkpoints/pre_cell_0.pickle

In [ ]:
%%RecordEvent
%%cudf.pandas.profile
### cell 0 ###





def load_lineitem(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/lineitem.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    print(df.columns)
    df["L_SHIPDATE"] = pd.to_datetime(df.L_SHIPDATE, format="%Y-%m-%d")
    df["L_RECEIPTDATE"] = pd.to_datetime(df.L_RECEIPTDATE, format="%Y-%m-%d")
    df["L_COMMITDATE"] = pd.to_datetime(df.L_COMMITDATE, format="%Y-%m-%d")
    return df
lineitem = load_lineitem(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
%Checkpoint /home/dias-benchmarks/tpch/notebooks/q09/q09_rewrite/checkpoints/pre_cell_1.pickle

In [ ]:
%%RecordEvent
%%cudf.pandas.profile
### cell 1 ###





def load_orders(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/orders.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    df["O_ORDERDATE"] = pd.to_datetime(df.O_ORDERDATE, format="%Y-%m-%d")
    return df
orders = load_orders(DATA_ROOT, **STORAGE_OPTS)


In [ ]:
%Checkpoint /home/dias-benchmarks/tpch/notebooks/q09/q09_rewrite/checkpoints/pre_cell_2.pickle

In [ ]:
%%RecordEvent
%%cudf.pandas.profile
### cell 2 ###




def load_part(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/part.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    return df
part = load_part(DATA_ROOT, **STORAGE_OPTS)


In [ ]:
%Checkpoint /home/dias-benchmarks/tpch/notebooks/q09/q09_rewrite/checkpoints/pre_cell_3.pickle

In [ ]:
%%RecordEvent
%%cudf.pandas.profile
### cell 3 ###





def load_nation(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/nation.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    return df
nation = load_nation(DATA_ROOT, **STORAGE_OPTS)


In [ ]:
%Checkpoint /home/dias-benchmarks/tpch/notebooks/q09/q09_rewrite/checkpoints/pre_cell_4.pickle

In [ ]:
%%RecordEvent
%%cudf.pandas.profile
### cell 4 ###





def load_partsupp(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/partsupp.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")  
    return df
partsupp = load_partsupp(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
%Checkpoint /home/dias-benchmarks/tpch/notebooks/q09/q09_rewrite/checkpoints/pre_cell_5.pickle

In [ ]:
%%RecordEvent
%%cudf.pandas.profile
### cell 5 ###




def load_supplier(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/supplier.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    return df
supplier = load_supplier(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
%Checkpoint /home/dias-benchmarks/tpch/notebooks/q09/q09_rewrite/checkpoints/pre_cell_6.pickle

In [ ]:
%%RecordEvent
%%cudf.pandas.profile
### cell 6 ###






psel = part.P_NAME.str.contains("ghost")
fpart = part[psel]
jn1 = lineitem.merge(fpart, left_on="L_PARTKEY", right_on="P_PARTKEY")
jn2 = jn1.merge(supplier, left_on="L_SUPPKEY", right_on="S_SUPPKEY")
jn3 = jn2.merge(nation, left_on="S_NATIONKEY", right_on="N_NATIONKEY")
jn4 = partsupp.merge(
    jn3, left_on=["PS_PARTKEY", "PS_SUPPKEY"], right_on=["L_PARTKEY", "L_SUPPKEY"]
)
jn5 = jn4.merge(orders, left_on="L_ORDERKEY", right_on="O_ORDERKEY")
jn5["TMP"] = jn5.L_EXTENDEDPRICE * (1 - jn5.L_DISCOUNT) - (
    (1 * jn5.PS_SUPPLYCOST) * jn5.L_QUANTITY
)
jn5["O_YEAR"] = jn5.O_ORDERDATE.dt.year
gb = jn5.groupby(["N_NAME", "O_YEAR"], as_index=False, sort=False)["TMP"].sum()
total = gb.sort_values(["N_NAME", "O_YEAR"], ascending=[True, False])

In [ ]:
%Checkpoint /home/dias-benchmarks/tpch/notebooks/q09/q09_rewrite/checkpoints/pre_cell_7.pickle

In [ ]:
%%RecordEvent
%%cudf.pandas.profile
### cell 7 ###

total.info()